# Squarepoint Capital DSI — EDA Interview Guide

> **Purpose:** A structured, battle-tested EDA playbook for quant research data interviews.  
> Every section includes the *why* alongside the *how* — because interviewers evaluate thinking, not pandas syntax.

---

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Stage 1 — First Look (5 min)](#2-stage-1--first-look-5-min)
3. [Stage 2 — Distributions & Statistics](#3-stage-2--distributions--statistics)
4. [Stage 3 — Data Quality Audit](#4-stage-3--data-quality-audit)
5. [Stage 4 — Temporal Structure](#5-stage-4--temporal-structure)
6. [Stage 5 — Relationships & Signals](#6-stage-5--relationships--signals)
7. [Stage 6 — Baseline Model](#7-stage-6--baseline-model)
8. [Stage 7 — Communication Template](#8-stage-7--communication-template)
9. [Interview Cheatsheet](#9-interview-cheatsheet)

---

**Key principle:** Every code block should be followed by a verbal interpretation.  
Pattern to internalize: *"This tells us that... / This is surprising because... / This suggests we should..."*

---
## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, normaltest, jarque_bera
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})
SEED = 42
np.random.seed(SEED)

print('✓ All imports OK')

In [ ]:
# ── Synthetic dataset: daily OHLCV + factor data for N assets ────────────────
# Replace this block with: df = pd.read_csv('your_dataset.csv', parse_dates=['date'], index_col='date')

np.random.seed(SEED)
n_days   = 504   # ~2 trading years
n_assets = 5
assets   = [f'ASSET_{i:02d}' for i in range(n_assets)]
dates    = pd.bdate_range('2022-01-01', periods=n_days)

rows = []
for asset in assets:
    log_price = np.cumsum(np.random.normal(0.0003, 0.015, n_days))
    price     = 100 * np.exp(log_price)
    volume    = np.random.lognormal(10, 1, n_days)
    signal    = np.random.normal(0, 1, n_days)  # synthetic alpha factor

    # Inject realistic artifacts
    stale_idx = np.random.choice(n_days - 5, 3)
    for idx in stale_idx:
        price[idx:idx+4] = price[idx]  # stale price

    nan_idx = np.random.choice(n_days, 10)
    price[nan_idx] = np.nan             # missing values
    volume[np.random.choice(n_days, 5)] = 0  # zero volume

    for i, d in enumerate(dates):
        rows.append({
            'date': d, 'asset': asset,
            'price': price[i], 'volume': volume[i], 'signal': signal[i]
        })

df_long = pd.DataFrame(rows)
df = df_long.pivot(index='date', columns='asset', values='price')
df_vol  = df_long.pivot(index='date', columns='asset', values='volume')
df_sig  = df_long.pivot(index='date', columns='asset', values='signal')

print(f'Dataset shape: {df.shape}  ({df.shape[0]} days × {df.shape[1]} assets)')

---
## 2. Stage 1 — First Look (5 min)

**Goal:** Understand *what* the data is before touching any numbers.  
Verbal cue: *"Before I do anything else, I want to get a feel for the shape, types, and obvious issues."*

In [ ]:
# ── The mandatory first 6 commands ──────────────────────────────────────────
print('── Shape ──────────────────────────────')
print(df.shape)

print('\n── Data types ─────────────────────────')
print(df.dtypes)

print('\n── Index ──────────────────────────────')
print(f'Type: {type(df.index).__name__}')
print(f'Range: {df.index.min()}  →  {df.index.max()}')
print(f'Freq inferred: {pd.infer_freq(df.index)}')

In [ ]:
print('── Head ────────────────────────────────')
display(df.head(8))

print('── Tail ────────────────────────────────')
display(df.tail(8))

In [ ]:
# ── Missing values — first pass ──────────────────────────────────────────────
missing_abs  = df.isna().sum()
missing_pct  = (df.isna().sum() / len(df) * 100).round(2)

pd.DataFrame({'missing_count': missing_abs, 'missing_%': missing_pct})

**Interpretation template:**
> *"Shape is reasonable. Index is a proper DatetimeIndex with business-day frequency — that's a good sign. I see ~2% missing values in some assets which is typical for illiquid names around holidays. I'll investigate the pattern in the Data Quality section."*

---
## 3. Stage 2 — Distributions & Statistics

**Goal:** Understand the *shape* of the data — fat tails, skew, outliers.  
Verbal cue: *"In financial data I always check for heavy tails — they matter for risk metrics."*

In [ ]:
# ── Extended describe — note the tail percentiles ───────────────────────────
df.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)

In [ ]:
# ── Always work with returns, not prices ────────────────────────────────────
returns = df.pct_change().dropna()
log_returns = np.log(df / df.shift(1)).dropna()

print('Returns describe (tail percentiles):')
returns.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(4)

In [ ]:
# ── Distribution analysis: normality tests + higher moments ─────────────────
print(f"{'Asset':<12} {'Skew':>8} {'Kurtosis':>10} {'JB p-val':>10} {'Interpretation'}")
print('-' * 65)

for col in returns.columns:
    r = returns[col].dropna()
    skew  = r.skew()
    kurt  = r.kurtosis()   # excess kurtosis (normal = 0)
    jb_p  = jarque_bera(r)[1]
    note  = 'heavy tails' if kurt > 3 else ('fat tails' if kurt > 1 else 'near-normal')
    print(f"{col:<12} {skew:>8.3f} {kurt:>10.3f} {jb_p:>10.4f}  {note}")

In [ ]:
# ── Visual: prices + return distributions ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Price levels
df.plot(ax=axes[0, 0], legend=True, linewidth=1)
axes[0, 0].set_title('Price levels')
axes[0, 0].set_ylabel('Price')

# Log prices (stationarize the trend visually)
np.log(df).plot(ax=axes[0, 1], legend=True, linewidth=1)
axes[0, 1].set_title('Log prices  (trend visible)')

# Return distributions vs normal
for col in returns.columns:
    returns[col].hist(ax=axes[1, 0], bins=50, alpha=0.4, density=True, label=col)
x = np.linspace(returns.min().min(), returns.max().max(), 200)
axes[1, 0].plot(x, stats.norm.pdf(x, returns.stack().mean(), returns.stack().std()),
                'k--', linewidth=1.5, label='Normal')
axes[1, 0].set_title('Return distributions vs Normal')
axes[1, 0].legend(fontsize=8)

# Rolling volatility — check for clustering
(returns.rolling(21).std() * np.sqrt(252)).plot(ax=axes[1, 1], legend=True, linewidth=1)
axes[1, 1].set_title('Rolling 21-day annualized volatility')
axes[1, 1].set_ylabel('Annualized vol')

plt.tight_layout()
plt.show()

**Interpretation template:**
> *"Returns show excess kurtosis — heavier tails than Normal. Jarque-Bera rejects normality. This is expected for financial data, but it means I should be careful with any model that assumes Gaussian errors. The rolling vol chart shows clear volatility clustering — GARCH-type effects — which is a well-known feature of equity returns."*

---
## 4. Stage 3 — Data Quality Audit

**Goal:** Actively hunt for artifacts.  
Verbal cue: *"I think about what specific artifacts are plausible given this data type, then check for each one."*

For market data the checklist is:
- Stale prices (price unchanged for many periods)
- Zero volume with non-zero price
- Temporal gaps (missing trading days)
- Extreme returns (fat finger errors)
- Crossed bid-ask (if applicable)
- Duplicate timestamps
- NaN patterns (random vs systematic)

In [ ]:
# ── 1. Duplicate timestamps ──────────────────────────────────────────────────
dups = df.index.duplicated().sum()
print(f'Duplicate timestamps: {dups}')

In [ ]:
# ── 2. Temporal gaps ─────────────────────────────────────────────────────────
time_diffs = pd.Series(df.index).diff().dropna()
expected   = time_diffs.median()
large_gaps = time_diffs[time_diffs > expected * 3]

print(f'Expected frequency: {expected}')
print(f'Non-standard gaps found: {len(large_gaps)}')
if len(large_gaps) > 0:
    for i, (idx, gap) in enumerate(large_gaps.items()):
        print(f'  Gap at {df.index[idx]}: {gap}')

In [ ]:
# ── 3. Stale prices ──────────────────────────────────────────────────────────
# A price that doesn't move for N consecutive periods is suspicious
STALE_WINDOW = 5

stale_report = {}
for col in df.columns:
    price_diff = df[col].diff().abs()
    consecutive_zeros = (price_diff == 0).astype(int)
    stale_count = (consecutive_zeros.rolling(STALE_WINDOW).sum() >= STALE_WINDOW - 1).sum()
    stale_report[col] = stale_count

stale_df = pd.Series(stale_report, name='stale_periods').to_frame()
stale_df['is_issue'] = stale_df['stale_periods'] > 0
print(f'Stale price check (window={STALE_WINDOW}):')
display(stale_df)

In [ ]:
# ── 4. Zero volume with non-zero price change ────────────────────────────────
price_moved  = df.pct_change().abs() > 0.001
zero_vol     = df_vol == 0
suspicious   = (price_moved & zero_vol).sum()

print('Zero volume + non-zero price change (suspicious):')
print(suspicious.to_string())

In [ ]:
# ── 5. Extreme returns — fat finger / bad tick detection ─────────────────────
EXTREME_THRESHOLD = 0.15  # >15% in one period

extreme_returns = (returns.abs() > EXTREME_THRESHOLD)
print(f'Returns exceeding {EXTREME_THRESHOLD*100:.0f}% threshold:')
print(extreme_returns.sum().to_string())

# Show the actual extreme events
if extreme_returns.any().any():
    for col in returns.columns:
        extremes = returns[col][extreme_returns[col]]
        if len(extremes) > 0:
            print(f'\n{col}:')
            print(extremes.round(4).to_string())

In [ ]:
# ── 6. NaN pattern analysis ──────────────────────────────────────────────────
# Random NaN = sensor failure / no trade
# Systematic NaN (same dates across assets) = data provider issue or holiday

nan_mask = df.isna()
simultaneous_nans = nan_mask.sum(axis=1)  # how many assets are NaN on each day

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of NaN positions
sns.heatmap(nan_mask.T, ax=axes[0], cbar=False,
            cmap='Reds', xticklabels=False, yticklabels=True)
axes[0].set_title('NaN positions (red = missing)')
axes[0].set_xlabel('Time →')

# Distribution of simultaneous NaNs
simultaneous_nans.value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Number of assets with NaN on same day')
axes[1].set_xlabel('Count of simultaneous NaNs')
axes[1].set_ylabel('Number of days')

plt.tight_layout()
plt.show()

print(f'Days where ALL assets have NaN: {(simultaneous_nans == df.shape[1]).sum()}')
print(f'Days where >50% assets have NaN: {(simultaneous_nans > df.shape[1]*0.5).sum()}')

**Interpretation template:**
> *"Found 3 stale-price episodes in ASSET_01 and ASSET_03 — likely illiquid periods. NaN pattern shows some correlated missingness across assets on the same dates, which suggests a data provider outage or holiday gap rather than individual asset issues. I'd flag these before any modeling and decide whether to forward-fill or exclude those dates."*

---
## 5. Stage 4 — Temporal Structure

**Goal:** Find *patterns in time* — seasonality, regime changes, stationarity.  
Verbal cue: *"Time-series data has structure that cross-sectional statistics miss entirely."*

In [ ]:
# ── Stationarity test — ADF ───────────────────────────────────────────────────
# Prices: non-stationary (unit root)
# Log-returns: should be stationary

print(f"{'Series':<20} {'ADF stat':>10} {'p-value':>10} {'Stationary?'}")
print('-' * 55)

for col in df.columns[:3]:  # prices
    series = df[col].dropna()
    adf_stat, p_val, *_ = adfuller(series)
    is_stat = '✓ Yes' if p_val < 0.05 else '✗ No'
    print(f"{col+' (price)':<20} {adf_stat:>10.3f} {p_val:>10.4f}  {is_stat}")

print()
for col in log_returns.columns[:3]:  # returns
    series = log_returns[col].dropna()
    adf_stat, p_val, *_ = adfuller(series)
    is_stat = '✓ Yes' if p_val < 0.05 else '✗ No'
    print(f"{col+' (log-ret)':<20} {adf_stat:>10.3f} {p_val:>10.4f}  {is_stat}")

In [ ]:
# ── Autocorrelation analysis ─────────────────────────────────────────────────
# ACF on returns: check for predictable serial structure (potential signal!)
# ACF on squared returns: check for volatility clustering (ARCH effects)

asset = df.columns[0]
r = log_returns[asset].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(r, ax=axes[0], lags=30, alpha=0.05)
axes[0].set_title(f'{asset}: ACF of log-returns\n(signal in serial structure?)')

plot_acf(r**2, ax=axes[1], lags=30, alpha=0.05)
axes[1].set_title(f'{asset}: ACF of squared returns\n(ARCH effects / vol clustering?)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Day-of-week seasonality ───────────────────────────────────────────────────
dow_returns = returns.copy()
dow_returns['day_of_week'] = returns.index.dayofweek
dow_returns['day_name']    = returns.index.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Mean return by day
day_mean = dow_returns.groupby('day_name')[df.columns].mean().reindex(day_order)
day_mean.mean(axis=1).plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Average return by day of week')
axes[0].set_ylabel('Mean return')
axes[0].tick_params(axis='x', rotation=30)

# Mean vol by day
day_vol = dow_returns.groupby('day_name')[df.columns].std().reindex(day_order)
day_vol.mean(axis=1).plot(kind='bar', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Average volatility by day of week')
axes[1].set_ylabel('Mean daily std')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Regime detection via rolling statistics ───────────────────────────────────
# Look for structural breaks: rolling mean and vol shifts

asset = df.columns[0]
r = log_returns[asset].dropna()

roll_mean = r.rolling(63).mean()   # ~quarterly
roll_vol  = r.rolling(63).std() * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

r.plot(ax=axes[0], linewidth=0.7, color='steelblue', alpha=0.8)
axes[0].set_title(f'{asset}: log-returns')

roll_mean.plot(ax=axes[1], color='darkblue', linewidth=1.5)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Rolling 63-day mean return')

roll_vol.plot(ax=axes[2], color='darkred', linewidth=1.5)
axes[2].set_title('Rolling 63-day annualized volatility')
axes[2].set_ylabel('Ann. vol')

plt.tight_layout()
plt.show()

**Interpretation template:**
> *"Prices are non-stationary as expected — ADF fails to reject the unit root. Log-returns are stationary. ACF on squared returns shows significant positive autocorrelation (ARCH effects) — volatility clusters. Returns ACF shows no strong serial structure, which means simple momentum on this timeframe isn't obvious. Rolling stats show a potential vol regime shift around mid-period."*

---
## 6. Stage 5 — Relationships & Signals

**Goal:** Move from *description* to *finding signals*.  
Verbal cue: *"Now I want to understand whether there's any predictive structure — and measure it rigorously."*

In [ ]:
# ── Cross-sectional correlation matrix ───────────────────────────────────────
corr = returns.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            annot_kws={'size': 9})
axes[0].set_title('Return correlation matrix')

# Correlation eigenvalue spectrum (check for factor structure)
eigvals = np.linalg.eigvalsh(corr)[::-1]
axes[1].bar(range(1, len(eigvals)+1), eigvals, color='steelblue', alpha=0.7)
axes[1].axhline(1.0, color='red', linestyle='--', linewidth=1, label='Random (Marchenko-Pastur ≈ 1)')
axes[1].set_title('Eigenvalue spectrum of correlation matrix\n(>1 suggests real factors)')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Eigenvalue')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Lagged correlation — is signal predictive? ────────────────────────────────
# For each lag k: corr(signal_t, return_{t+k})

lags         = [1, 2, 3, 5, 10, 21]
lag_corr_raw = []  # Pearson
lag_ic       = []  # Spearman (IC — the correct quant metric)

asset = df.columns[0]
sig   = df_sig[asset]
ret   = log_returns[asset]

for lag in lags:
    fwd_ret = ret.shift(-lag)          # forward return
    valid   = sig.notna() & fwd_ret.notna()
    s, r    = sig[valid], fwd_ret[valid]

    pearson  = s.corr(r)
    spear, p = spearmanr(s, r)
    lag_corr_raw.append({'lag': lag, 'pearson': pearson, 'spearman_IC': spear, 'IC_pval': p})

lag_df = pd.DataFrame(lag_corr_raw).set_index('lag')
print(f'Signal predictability for {asset}:')
display(lag_df.round(4))

In [ ]:
# ── Rolling IC — is the signal stable over time? ─────────────────────────────
# IC should be consistently positive, not just good in one period

FORWARD_LAG  = 5   # predict 5-day forward return
ROLLING_WIN  = 63  # ~quarterly

fwd_ret = log_returns[asset].shift(-FORWARD_LAG)
aligned = pd.concat([df_sig[asset].rename('signal'), fwd_ret.rename('fwd_ret')], axis=1).dropna()

rolling_ic = aligned.rolling(ROLLING_WIN).apply(
    lambda x: spearmanr(x[:, 0], x[:, 1])[0], raw=True
)['signal']   # take the signal column result

fig, ax = plt.subplots(figsize=(14, 4))
rolling_ic.plot(ax=ax, linewidth=1.2, color='darkblue', label='Rolling IC')
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.axhline(rolling_ic.mean(), color='red', linestyle=':', linewidth=1.2,
           label=f'Mean IC = {rolling_ic.mean():.4f}')
ax.fill_between(rolling_ic.index, rolling_ic, 0,
                where=rolling_ic > 0, alpha=0.15, color='green', label='Positive IC')
ax.fill_between(rolling_ic.index, rolling_ic, 0,
                where=rolling_ic < 0, alpha=0.15, color='red', label='Negative IC')
ax.set_title(f'Rolling {ROLLING_WIN}-day IC: signal → {FORWARD_LAG}-day forward return')
ax.set_ylabel('IC (Spearman)')
ax.legend()
plt.tight_layout()
plt.show()

ic_positive_pct = (rolling_ic > 0).mean()
print(f'Mean IC:         {rolling_ic.mean():.4f}')
print(f'IC std (ICIR):   {rolling_ic.std():.4f}   →  ICIR = {rolling_ic.mean()/rolling_ic.std():.4f}')
print(f'IC > 0:          {ic_positive_pct:.1%} of periods')

In [ ]:
# ── Quintile analysis — monotonic return spread? ─────────────────────────────
# If signal is good: Q5 (top quintile) > Q1 (bottom quintile)

aligned['quintile'] = pd.qcut(aligned['signal'], 5, labels=[1, 2, 3, 4, 5])
quintile_stats = aligned.groupby('quintile')['fwd_ret'].agg(['mean', 'std', 'count'])
quintile_stats.columns = ['mean_return', 'volatility', 'count']
quintile_stats['sharpe_approx'] = (quintile_stats['mean_return'] /
                                    quintile_stats['volatility'] * np.sqrt(252))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(quintile_stats.index.astype(str),
              quintile_stats['mean_return'] * 10000,
              color=['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4'],
              alpha=0.85, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Mean forward return by signal quintile\n(monotonic spread = signal works)')
ax.set_xlabel('Quintile (1 = low signal, 5 = high signal)')
ax.set_ylabel('Mean forward return (bps)')
plt.tight_layout()
plt.show()

print('\nQuintile statistics:')
display(quintile_stats.round(6))

spread = (quintile_stats.loc[5, 'mean_return'] - quintile_stats.loc[1, 'mean_return']) * 10000
print(f'\nQ5 - Q1 spread: {spread:.2f} bps')

**Interpretation template:**
> *"Mean IC is ~0.03 with ICIR of ~0.4 — modest but not zero. More importantly, the rolling IC shows it's fairly stable over time rather than driven by one lucky period. The quintile spread shows a near-monotonic pattern: Q5 outperforms Q1 by X bps on average. This is a promising initial signal, but I'd want to check it after transaction costs and in different vol regimes."*

---
## 7. Stage 6 — Baseline Model

**Goal:** Build *something that works* within the time constraints.  
Verbal cue: *"I never use a standard train-test split for financial data — that creates look-ahead bias. I always use walk-forward validation to simulate real production conditions."*

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────

asset = df.columns[0]
r     = log_returns[asset].dropna()
sig   = df_sig[asset]

features = pd.DataFrame(index=r.index)

# Price-based features (momentum at multiple horizons)
for h in [5, 10, 21, 63]:
    features[f'mom_{h}d'] = r.rolling(h).sum()

# Volatility features (vol normalization)
features['vol_5d']  = r.rolling(5).std()
features['vol_21d'] = r.rolling(21).std()
features['vol_ratio'] = features['vol_5d'] / features['vol_21d']  # vol compression?

# The alpha signal
features['signal'] = sig

# Target: 5-day forward return
FORWARD_HORIZON = 5
target = r.shift(-FORWARD_HORIZON).rename('fwd_ret')

# Align and clean
data = pd.concat([features, target], axis=1).dropna()
feature_cols = features.columns.tolist()

print(f'Clean dataset: {data.shape}')
print(f'Features: {feature_cols}')

In [ ]:
# ── Walk-forward validation ───────────────────────────────────────────────────
# This is the ONLY acceptable validation approach for financial time-series

N          = len(data)
TRAIN_FRAC = 0.6
STEP       = 21      # re-fit monthly
MIN_TRAIN  = int(N * TRAIN_FRAC)

X = data[feature_cols].values
y = data['fwd_ret'].values

results = []

for start in range(MIN_TRAIN, N - STEP, STEP):
    X_tr, y_tr = X[:start], y[:start]
    X_te, y_te = X[start:start+STEP], y[start:start+STEP]

    # Scale within each window (no future leakage)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    model = Ridge(alpha=1.0)
    model.fit(X_tr_s, y_tr)
    preds = model.predict(X_te_s)

    ic, p_val = spearmanr(preds, y_te)
    results.append({
        'date':  data.index[start],
        'IC':    ic,
        'p_val': p_val,
        'MSE':   mean_squared_error(y_te, preds),
    })

res_df = pd.DataFrame(results).set_index('date')

print(f"Walk-forward results ({len(res_df)} test periods):")
print(f"  Mean IC:         {res_df['IC'].mean():.4f}")
print(f"  IC std:          {res_df['IC'].std():.4f}")
print(f"  ICIR:            {res_df['IC'].mean() / res_df['IC'].std():.4f}")
print(f"  IC > 0:          {(res_df['IC'] > 0).mean():.1%} of windows")

In [ ]:
# ── Plot walk-forward IC over time ────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

res_df['IC'].plot(ax=axes[0], linewidth=1.2, color='steelblue', label='IC per window')
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].axhline(res_df['IC'].mean(), color='red', linestyle=':', linewidth=1.2,
                label=f"Mean IC = {res_df['IC'].mean():.4f}")
axes[0].fill_between(res_df.index, res_df['IC'], 0,
                     where=res_df['IC'] > 0, alpha=0.2, color='green')
axes[0].fill_between(res_df.index, res_df['IC'], 0,
                     where=res_df['IC'] < 0, alpha=0.2, color='red')
axes[0].set_title('Walk-forward IC over time')
axes[0].legend()

res_df['IC'].cumsum().plot(ax=axes[1], linewidth=1.5, color='darkblue')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Cumulative IC (upward slope = consistent signal)')
axes[1].set_ylabel('Cumulative IC')

plt.tight_layout()
plt.show()

---
## 8. Stage 7 — Communication Template

**Goal:** Structure your findings into a coherent story.  
Verbal cue: *"Let me summarize what I found, what works, what's surprising, and what I'd do next."*

In [ ]:
# ── Auto-generate summary statistics ──────────────────────────────────────────

summary = {
    'Dataset':           f"{df.shape[0]} days × {df.shape[1]} assets",
    'Date range':        f"{df.index.min().date()} → {df.index.max().date()}",
    'Missing values':    f"{df.isna().mean().mean():.1%} average across assets",
    'Stale price flags': f"{sum(stale_report.values())} periods across all assets",
    'Temporal gaps':     f"{len(large_gaps)} non-standard gaps",
    'Excess kurtosis':   f"{returns.kurtosis().mean():.2f} (avg across assets)",
    'Signal mean IC':    f"{res_df['IC'].mean():.4f}",
    'ICIR':              f"{res_df['IC'].mean() / res_df['IC'].std():.4f}",
    'IC > 0 frequency':  f"{(res_df['IC'] > 0).mean():.1%}",
    'Q5-Q1 spread':      f"{spread:.2f} bps",
}

print('='*55)
print('EDA SUMMARY')
print('='*55)
for k, v in summary.items():
    print(f'  {k:<25} {v}')
print('='*55)

### Story Structure for Verbal Communication

Use this 3-part structure for your final verbal summary:

---

**Part 1 — What is this data?**
> *"This is daily price and volume data for N assets over X years. Data quality is overall [good/acceptable/problematic]. I found [stale prices / temporal gaps / NaN patterns] which are [typical for this type / worth investigating]. I cleaned them by [forward-fill / exclusion] before modeling."*

**Part 2 — What did I find?**
> *"Returns are non-normal with heavy tails and volatility clustering — standard for equities. The signal shows a mean IC of [X] with ICIR of [Y], which is [modest / strong] but consistent — IC is positive in [Z]% of rolling windows. The quintile spread confirms a roughly monotonic relationship: top quintile outperforms bottom by [X] bps on average."*

**Part 3 — What would I do next?**
> *"Three directions: (1) Check whether the signal holds after realistic transaction costs — the spread looks attractive pre-costs but could compress. (2) Test industry- and vol-neutralized versions of the factor to confirm the signal isn't just a proxy for sector exposure. (3) Explore non-linear models — LightGBM — to see if the quintile relationship is actually non-linear."*

---

**What NOT to say:**
- *"I ran describe() and everything looks fine."* — No interpretation.
- *"The correlation is 0.03."* — So what? What does it mean for trading?
- *"I tried a neural network and got 85% accuracy."* — Wrong framework for returns.

**What TO say:**
- *"This is surprising because..."*
- *"This tells us that..."*
- *"I'd be careful about this because..."*
- *"If I had more time I would..."*

---
## 9. Interview Cheatsheet

### Time allocation for a full-day DSI

| Stage | Focus | Target time |
|---|---|---|
| 1. First look | shape, dtypes, head, missing | ~5 min |
| 2. Distributions | describe, histograms, kurtosis | ~10 min |
| 3. Quality audit | stale, gaps, dupes, anomalies | ~15 min |
| 4. Temporal structure | seasonality, ACF, stationarity | ~15 min |
| 5. Signals | lagged corr, IC, quintile analysis | ~20 min |
| 6. Baseline model | walk-forward Ridge / LightGBM | ~25 min |
| 7. Summary | story + limitations + next steps | ~10 min |

### Common pitfalls

| Pitfall | Why it's wrong | Correct approach |
|---|---|---|
| `train_test_split` on time-series | Leaks future into past | Walk-forward validation |
| Pearson correlation for signals | Sensitive to outliers | Spearman IC |
| Working with price levels | Non-stationary | Work with log-returns |
| `describe()` without interpretation | Says nothing | Explain what you see |
| Fitting and transforming on full data | Look-ahead bias | Fit scaler on train window only |
| R² as the metric | Useless for returns | IC, ICIR, Sharpe |

### Useful one-liners to say during the interview

- *"I'm going to use Spearman rather than Pearson here — returns have heavy tails and outliers would distort a linear correlation."*
- *"I always check autocorrelation of squared returns — significant ACF there means volatility clustering, which affects my vol estimates."*
- *"I'm fitting the scaler on the training window only and applying it to the test window — otherwise there's data leakage."*
- *"An IC of 0.03 sounds small but is actually a usable signal at scale — the question is whether it survives transaction costs."*
- *"I'd want to check if this is still there after neutralizing for market beta and sector."*